# retention_candidate_overlay_screen_v1

Executable diagnostic for the strict SOXL actor-retention goal.

**Baselines:** canonical strict gate from `model_retention_walkforward_gate_v1`, SOXLBacktest qlib buy/sell reference `soxl_backtest_policy_reimplementation_v1`, direct comparison source `soxlbacktest_reference_model_score_v1`.

**Role:** diagnose. This notebook does not promote an actor; it screens whether existing full/WF daily exports and simple position-path overlays can identify a target worth retraining.

**Target window:** YTD `2026-01-02..2026-06-09` plus required WF windows `wf_2020`, `wf_2022`, `wf_2023_2024`.

**Changed layer:** model_selection / training-target screen.

**Hypothesis:** an existing complete-WF export or simple prior-day position overlay may reveal a candidate target that keeps WF MaxDD below 20% while preserving enough YTD return.

**Historical success/failure context:** closest success is SOXLBacktest qlib reference for YTD return and low-risk variants for drawdown, but no SOXLBacktest variant passes both. Closest failure is `return_preserving_hybrid_target_bc_replay_v1`: two DD gates and two relative gates passed, but no complete fold gate. This run is not a duplicate because it re-screens complete daily exports under the current strict final gate and records whether a simple overlay target is worth training.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "AGENTS.md").exists() and (parent / "src/soxl_quant").exists()
)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from soxl_quant.app.retention_candidate_screen import (
    OverlaySpec,
    candidate_to_row,
    evaluate_export_nav_path,
    evaluate_position_path,
    load_daily_path,
    strict_gate_failures,
    summarize_candidate,
)

CARD_DIR = ROOT / "runtime/model_selection/card/retention_candidate_overlay_screen_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

GATE_SUMMARY = json.loads((ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json").read_text())["summary"]
SCORE_SUMMARY = pd.read_csv(ROOT / "runtime/model_selection/card/soxlbacktest_reference_model_score_v1/candidate_score_summary.csv")

REQUIRED_WF_WINDOWS = tuple(GATE_SUMMARY["required_wf_windows"])
MAX_WF_DD = float(GATE_SUMMARY["max_walkforward_drawdown_pct"])
REQUIRED_SOXLBT_RETURN = float(GATE_SUMMARY["reference_return_pct"])
REQUIRED_BEST_RETURN = float(GATE_SUMMARY["required_best_return_pct"])
YTD_START = "2026-01-02"
YTD_END = "2026-06-09"

thresholds = {
    "required_wf_windows": REQUIRED_WF_WINDOWS,
    "max_walkforward_drawdown_pct": MAX_WF_DD,
    "required_soxlbacktest_return_pct": REQUIRED_SOXLBT_RETURN,
    "required_best_return_pct": REQUIRED_BEST_RETURN,
    "ytd_start": YTD_START,
    "ytd_end": YTD_END,
}
thresholds

{'required_wf_windows': ('wf_2020', 'wf_2022', 'wf_2023_2024'),
 'max_walkforward_drawdown_pct': 20.0,
 'required_soxlbacktest_return_pct': 132.077978,
 'required_best_return_pct': 213.143462,
 'ytd_start': '2026-01-02',
 'ytd_end': '2026-06-09'}

In [2]:
def daily_paths_for_profile(profile: str) -> dict[str, Path]:
    base = ROOT / "runtime/model/card" / profile
    paths = {"ytd": base / f"{profile}_soxl_daily.csv"}
    for window in REQUIRED_WF_WINDOWS:
        paths[window] = base / f"{profile}_{window}_soxl_daily.csv"
    return paths

complete_sources = []
for row in SCORE_SUMMARY.to_dict("records"):
    profile = str(row["profile"])
    if not bool(row.get("complete_wf", False)):
        continue
    paths = daily_paths_for_profile(profile)
    if all(path.exists() for path in paths.values()):
        complete_sources.append(profile)

source_manifest = []
for profile in complete_sources:
    paths = daily_paths_for_profile(profile)
    source_manifest.append({"profile": profile, **{key: str(path.relative_to(ROOT)) for key, path in paths.items()}})

source_manifest_df = pd.DataFrame(source_manifest)
source_manifest_df.to_csv(CARD_DIR / "source_daily_manifest.csv", index=False)
source_manifest_df

,profile,ytd,wf_2020,wf_2022,wf_2023_2024
0,state_conditioned_drawdown_rebound_reward_trai...,runtime/model/card/state_conditioned_drawdown_...,runtime/model/card/state_conditioned_drawdown_...,runtime/model/card/state_conditioned_drawdown_...,runtime/model/card/state_conditioned_drawdown_...
1,drawdown_rebound_reward_train_v1,runtime/model/card/drawdown_rebound_reward_tra...,runtime/model/card/drawdown_rebound_reward_tra...,runtime/model/card/drawdown_rebound_reward_tra...,runtime/model/card/drawdown_rebound_reward_tra...
2,mode_conditioned_rebound_reward_train_v2,runtime/model/card/mode_conditioned_rebound_re...,runtime/model/card/mode_conditioned_rebound_re...,runtime/model/card/mode_conditioned_rebound_re...,runtime/model/card/mode_conditioned_rebound_re...


In [3]:
overlay_specs = [OverlaySpec(name="base_position_path", base_scale=1.0, global_cap=1.0, mild_cap=1.0, severe_cap=1.0)]
for scale in [0.90, 1.00, 1.05, 1.10, 1.15, 1.20, 1.30, 1.40]:
    for global_cap in [0.45, 0.55, 0.65, 0.75, 0.85, 1.00]:
        overlay_specs.append(
            OverlaySpec(
                name=f"scale{scale:.2f}_cap{global_cap:.2f}",
                base_scale=scale,
                global_cap=global_cap,
                mild_cap=global_cap,
                severe_cap=global_cap,
            )
        )
for scale in [1.00, 1.10, 1.20, 1.30, 1.40]:
    for mild_cap, severe_cap in [(0.45, 0.25), (0.40, 0.20), (0.35, 0.15), (0.30, 0.10)]:
        overlay_specs.append(
            OverlaySpec(
                name=f"scale{scale:.2f}_dd63_m{mild_cap:.2f}_s{severe_cap:.2f}",
                base_scale=scale,
                global_cap=1.0,
                drawdown_lookback=63,
                mild_drawdown=-0.10,
                mild_cap=mild_cap,
                severe_drawdown=-0.20,
                severe_cap=severe_cap,
            )
        )
len(overlay_specs)

69

In [4]:
def filter_dates(frame: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    return frame[(frame["datetime"] >= start) & (frame["datetime"] <= end)].reset_index(drop=True)

def gate_loss(row: dict[str, object]) -> float:
    missing = str(row.get("missing_wf_windows") or "")
    missing_count = 0 if missing in {"", "nan", "None"} else len([part for part in missing.split(",") if part])
    ytd = row.get("ytd_return_pct")
    worst = row.get("worst_wf_max_drawdown_pct")
    ytd_gap = max(REQUIRED_BEST_RETURN - float(ytd), 0.0) if ytd is not None else 1000.0
    soxlbt_gap = max(REQUIRED_SOXLBT_RETURN - float(ytd), 0.0) if ytd is not None else 1000.0
    dd_gap = max(float(worst) - MAX_WF_DD, 0.0) if worst is not None else 1000.0
    return round(missing_count * 1000.0 + dd_gap * 20.0 + ytd_gap + soxlbt_gap * 0.5, 6)

rows = []
window_rows = []
for profile in complete_sources:
    paths = daily_paths_for_profile(profile)
    ytd_frame = filter_dates(load_daily_path(paths["ytd"]), YTD_START, YTD_END)
    wf_frames = {window: load_daily_path(paths[window]) for window in REQUIRED_WF_WINDOWS}

    source_ytd_metrics = evaluate_export_nav_path(ytd_frame)
    source_wf_metrics = {window: evaluate_export_nav_path(frame) for window, frame in wf_frames.items()}
    source_candidate = f"{profile}__source_export_nav"
    source_metrics = summarize_candidate(
        candidate=source_candidate,
        ytd_metrics=source_ytd_metrics,
        wf_metrics=source_wf_metrics,
        required_wf_windows=REQUIRED_WF_WINDOWS,
    )
    source_failures = strict_gate_failures(
        source_metrics,
        required_wf_windows=REQUIRED_WF_WINDOWS,
        max_walkforward_drawdown_pct=MAX_WF_DD,
        required_soxlbacktest_return_pct=REQUIRED_SOXLBT_RETURN,
        required_best_return_pct=REQUIRED_BEST_RETURN,
    )
    row = candidate_to_row(source_metrics, failures=source_failures, overlay="source_export_nav", source_profile=profile)
    row.update({"metric_source": "strategy_nav_export", "gate_loss": gate_loss(row), "base_scale": None, "global_cap": None, "mild_cap": None, "severe_cap": None})
    rows.append(row)
    window_rows.append({"candidate": source_candidate, "source_profile": profile, "overlay": "source_export_nav", "metric_source": "strategy_nav_export", "window": "ytd_2026_common", **source_ytd_metrics.__dict__})
    for window, metrics in source_wf_metrics.items():
        window_rows.append({"candidate": source_candidate, "source_profile": profile, "overlay": "source_export_nav", "metric_source": "strategy_nav_export", "window": window, **metrics.__dict__})

    for spec in overlay_specs:
        ytd_metrics = evaluate_position_path(ytd_frame, spec)
        wf_metrics = {window: evaluate_position_path(frame, spec) for window, frame in wf_frames.items()}
        candidate = f"{profile}__{spec.name}"
        candidate_metrics = summarize_candidate(
            candidate=candidate,
            ytd_metrics=ytd_metrics,
            wf_metrics=wf_metrics,
            required_wf_windows=REQUIRED_WF_WINDOWS,
        )
        failures = strict_gate_failures(
            candidate_metrics,
            required_wf_windows=REQUIRED_WF_WINDOWS,
            max_walkforward_drawdown_pct=MAX_WF_DD,
            required_soxlbacktest_return_pct=REQUIRED_SOXLBT_RETURN,
            required_best_return_pct=REQUIRED_BEST_RETURN,
        )
        row = candidate_to_row(candidate_metrics, failures=failures, overlay=spec.name, source_profile=profile)
        row.update({"metric_source": "position_replay_diagnostic", "gate_loss": gate_loss(row), "base_scale": spec.base_scale, "global_cap": spec.global_cap, "mild_cap": spec.mild_cap, "severe_cap": spec.severe_cap})
        rows.append(row)
        window_rows.append({"candidate": candidate, "source_profile": profile, "overlay": spec.name, "metric_source": "position_replay_diagnostic", "window": "ytd_2026_common", **ytd_metrics.__dict__})
        for window, metrics in wf_metrics.items():
            window_rows.append({"candidate": candidate, "source_profile": profile, "overlay": spec.name, "metric_source": "position_replay_diagnostic", "window": window, **metrics.__dict__})

grid = pd.DataFrame(rows).sort_values(["strict_gate_pass", "gate_loss", "worst_wf_max_drawdown_pct"], ascending=[False, True, True]).reset_index(drop=True)
window_grid = pd.DataFrame(window_rows)
grid.to_csv(CARD_DIR / "overlay_grid_summary.csv", index=False)
window_grid.to_csv(CARD_DIR / "overlay_window_metrics.csv", index=False)
grid.head(20)

,candidate,source_profile,overlay,ytd_return_pct,ytd_max_drawdown_pct,worst_wf_max_drawdown_pct,min_wf_return_pct,mean_wf_return_pct,wf_windows_present,missing_wf_windows,strict_gate_pass,failures,metric_source,gate_loss,base_scale,global_cap,mild_cap,severe_cap
0,drawdown_rebound_reward_train_v1__source_expor...,drawdown_rebound_reward_train_v1,source_export_nav,172.832548,27.310377,20.934520,-7.000281,14.549028,"wf_2020,wf_2022,wf_2023_2024",,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...,strategy_nav_export,59.001314,NaN,NaN,NaN,NaN
1,state_conditioned_drawdown_rebound_reward_trai...,state_conditioned_drawdown_rebound_reward_trai...,source_export_nav,196.505054,31.390931,29.147985,-23.275008,10.113392,"wf_2020,wf_2022,wf_2023_2024",,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...,strategy_nav_export,199.598108,NaN,NaN,NaN,NaN
2,mode_conditioned_rebound_reward_train_v2__sour...,mode_conditioned_rebound_reward_train_v2,source_export_nav,177.573569,27.278641,30.489188,-16.085903,7.097381,"wf_2020,wf_2022,wf_2023_2024",,False,worst_wf_max_drawdown_ge_20;ytd_return_below_c...,strategy_nav_export,245.353653,NaN,NaN,NaN,NaN
3,state_conditioned_drawdown_rebound_reward_trai...,state_conditioned_drawdown_rebound_reward_trai...,scale1.40_dd63_m0.30_s0.10,0.495517,28.014839,18.679289,-17.908302,0.420421,"wf_2020,wf_2022,wf_2023_2024",,False,ytd_return_below_soxlbacktest_reference;ytd_re...,position_replay_diagnostic,278.439175,1.4,1.0,0.30,0.10
4,state_conditioned_drawdown_rebound_reward_trai...,state_conditioned_drawdown_rebound_reward_trai...,scale1.30_dd63_m0.30_s0.10,-0.183848,27.858741,18.301014,-17.526441,0.763074,"wf_2020,wf_2022,wf_2023_2024",,False,ytd_return_below_soxlbacktest_reference;ytd_re...,position_replay_diagnostic,279.458223,1.3,1.0,0.30,0.10
5,state_conditioned_drawdown_rebound_reward_trai...,state_conditioned_drawdown_rebound_reward_trai...,scale1.00_dd63_m0.30_s0.10,-0.582487,24.678146,17.161167,-16.375787,0.482533,"wf_2020,wf_2022,wf_2023_2024",,False,ytd_return_below_soxlbacktest_reference;ytd_re...,position_replay_diagnostic,280.056181,1.0,1.0,0.30,0.10
6,state_conditioned_drawdown_rebound_reward_trai...,state_conditioned_drawdown_rebound_reward_trai...,scale1.10_dd63_m0.30_s0.10,-0.751358,26.724146,17.541950,-16.760181,0.811906,"wf_2020,wf_2022,wf_2023_2024",,False,ytd_return_below_soxlbacktest_reference;ytd_re...,position_replay_diagnostic,280.309488,1.1,1.0,0.30,0.10
7,state_conditioned_drawdown_rebound_reward_trai...,state_conditioned_drawdown_rebound_reward_trai...,scale1.20_dd63_m0.30_s0.10,-0.872318,27.707123,17.921900,-17.143733,1.060156,"wf_2020,wf_2022,wf_2023_2024",,False,ytd_return_below_soxlbacktest_reference;ytd_re...,position_replay_diagnostic,280.490928,1.2,1.0,0.30,0.10
8,drawdown_rebound_reward_train_v1__scale1.00_dd...,drawdown_rebound_reward_train_v1,scale1.00_dd63_m0.35_s0.15,-1.574420,24.019360,19.880785,-4.774044,4.043588,"wf_2020,wf_2022,wf_2023_2024",,False,ytd_return_below_soxlbacktest_reference;ytd_re...,position_replay_diagnostic,281.544081,1.0,1.0,0.35,0.15
9,drawdown_rebound_reward_train_v1__scale1.00_dd...,drawdown_rebound_reward_train_v1,scale1.00_dd63_m0.30_s0.10,-1.614695,23.829918,15.540008,-4.562948,2.844512,"wf_2020,wf_2022,wf_2023_2024",,False,ytd_return_below_soxlbacktest_reference;ytd_re...,position_replay_diagnostic,281.604493,1.0,1.0,0.30,0.10


In [5]:
pass_count = int(grid["strict_gate_pass"].sum())
source_grid = grid[grid["metric_source"].eq("strategy_nav_export")].copy()
position_grid = grid[grid["metric_source"].eq("position_replay_diagnostic")].copy()
best = grid.iloc[0].to_dict() if len(grid) else {}
best_source = source_grid.sort_values("gate_loss").iloc[0].to_dict() if len(source_grid) else {}
best_position_replay = position_grid.sort_values("gate_loss").iloc[0].to_dict() if len(position_grid) else {}

summary = {
    "experiment": "retention_candidate_overlay_screen_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "overlay_target_screen_no_actor_promotion" if pass_count == 0 else "overlay_target_screen_found_strict_gate_candidate",
    "actor_retention_allowed": False,
    "promotion_allowed": False,
    "source_profiles": complete_sources,
    "source_export_candidate_count": int(len(source_grid)),
    "position_replay_overlay_count": len(overlay_specs),
    "candidate_count": int(len(grid)),
    "strict_gate_pass_count": pass_count,
    "thresholds": thresholds,
    "best_candidate": best,
    "best_source_export_candidate": best_source,
    "best_position_replay_diagnostic": best_position_replay,
    "position_replay_warning": "position-path replay is diagnostic only because exported position plus price does not reproduce strategy_nav returns for these actors",
    "sidecars": {
        "source_daily_manifest": str((CARD_DIR / "source_daily_manifest.csv").relative_to(ROOT)),
        "overlay_grid_summary": str((CARD_DIR / "overlay_grid_summary.csv").relative_to(ROOT)),
        "overlay_window_metrics": str((CARD_DIR / "overlay_window_metrics.csv").relative_to(ROOT)),
        "summary": str((CARD_DIR / "summary.json").relative_to(ROOT)),
        "conclusion": str((CARD_DIR / "conclusion.md").relative_to(ROOT)),
    },
}

if pass_count == 0:
    summary["next_action"] = "train_cap10_joint_contract_bc_checkpoint_with_explicit_wf2020_coverage_and_checkpoint_drift_stop; existing exports and simple overlays do not satisfy strict retention"
else:
    summary["next_action"] = "export candidate path as training target, rerun official actor replay, then apply strict retention gate before keep_actor"

(CARD_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
summary

{'experiment': 'retention_candidate_overlay_screen_v1',
 'created_at': '2026-06-14T04:44:23.122188+00:00',
 'decision': 'overlay_target_screen_no_actor_promotion',
 'actor_retention_allowed': False,
 'promotion_allowed': False,
 'source_profiles': ['state_conditioned_drawdown_rebound_reward_train_v1',
  'drawdown_rebound_reward_train_v1',
  'mode_conditioned_rebound_reward_train_v2'],
 'source_export_candidate_count': 3,
 'position_replay_overlay_count': 69,
 'candidate_count': 210,
 'strict_gate_pass_count': 0,
 'thresholds': {'required_wf_windows': ('wf_2020', 'wf_2022', 'wf_2023_2024'),
  'max_walkforward_drawdown_pct': 20.0,
  'required_soxlbacktest_return_pct': 132.077978,
  'required_best_return_pct': 213.143462,
  'ytd_start': '2026-01-02',
  'ytd_end': '2026-06-09'},
 'best_candidate': {'candidate': 'drawdown_rebound_reward_train_v1__source_export_nav',
  'source_profile': 'drawdown_rebound_reward_train_v1',
  'overlay': 'source_export_nav',
  'ytd_return_pct': 172.832548,
  'y

In [6]:
lines = [
    "# retention_candidate_overlay_screen_v1",
    "",
    f"- Decision: `{summary['decision']}`.",
    f"- Source profiles screened: `{', '.join(complete_sources)}`.",
    f"- Source export candidates: `{len(source_grid)}`; position replay overlays: `{len(position_grid)}`; strict gate pass count: `{pass_count}`.",
    f"- Gate: WF windows `{', '.join(REQUIRED_WF_WINDOWS)}`, worst WF MaxDD `< {MAX_WF_DD:.2f}%`, YTD return `>= {REQUIRED_SOXLBT_RETURN:.6f}%` and `>= {REQUIRED_BEST_RETURN:.6f}%`.",
]
if best_source:
    lines.extend([
        f"- Best official source export: `{best_source['candidate']}`.",
        f"- Best official source metrics: YTD `{float(best_source['ytd_return_pct']):.2f}%`, YTD MaxDD `{float(best_source['ytd_max_drawdown_pct']):.2f}%`, worst WF MaxDD `{float(best_source['worst_wf_max_drawdown_pct']):.2f}%`, failures `{best_source['failures'] or 'none'}`.",
    ])
if best_position_replay:
    lines.extend([
        f"- Best position-replay diagnostic: `{best_position_replay['candidate']}` with YTD `{float(best_position_replay['ytd_return_pct']):.2f}%` and worst WF MaxDD `{float(best_position_replay['worst_wf_max_drawdown_pct']):.2f}%`.",
        "- Position replay is diagnostic only: exported `position` plus SOXL price does not reproduce `strategy_nav`, so it cannot be used for retention or promotion scoring.",
    ])
if pass_count == 0:
    lines.append("- No existing complete-WF export or simple position-path overlay satisfies the final retention gate. This blocks actor retention and confirms the next useful work is actual cap10/joint-contract BC/checkpoint training with explicit WF coverage and drift stop, not a dashboard/registry workaround.")
else:
    lines.append("- At least one screened target satisfies the numeric screen. It still is not a retained actor; it must be converted into a trained/replayable actor and pass official strict retention before registry keep_actor can be restored.")
lines.extend([
    "",
    "## Sidecars",
    "- `runtime/model_selection/card/retention_candidate_overlay_screen_v1/source_daily_manifest.csv`",
    "- `runtime/model_selection/card/retention_candidate_overlay_screen_v1/overlay_grid_summary.csv`",
    "- `runtime/model_selection/card/retention_candidate_overlay_screen_v1/overlay_window_metrics.csv`",
    "- `runtime/model_selection/card/retention_candidate_overlay_screen_v1/summary.json`",
])
(CARD_DIR / "conclusion.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

# retention_candidate_overlay_screen_v1

- Decision: `overlay_target_screen_no_actor_promotion`.
- Source profiles screened: `state_conditioned_drawdown_rebound_reward_train_v1, drawdown_rebound_reward_train_v1, mode_conditioned_rebound_reward_train_v2`.
- Source export candidates: `3`; position replay overlays: `207`; strict gate pass count: `0`.
- Gate: WF windows `wf_2020, wf_2022, wf_2023_2024`, worst WF MaxDD `< 20.00%`, YTD return `>= 132.077978%` and `>= 213.143462%`.
- Best official source export: `drawdown_rebound_reward_train_v1__source_export_nav`.
- Best official source metrics: YTD `172.83%`, YTD MaxDD `27.31%`, worst WF MaxDD `20.93%`, failures `worst_wf_max_drawdown_ge_20;ytd_return_below_current_best_alignment`.
- Best position-replay diagnostic: `state_conditioned_drawdown_rebound_reward_train_v1__scale1.40_dd63_m0.30_s0.10` with YTD `0.50%` and worst WF MaxDD `18.68%`.
- Position replay is diagnostic only: exported `position` plus SOXL price does not reproduce `strate